# GEE GSMaP Downloader - Optimized Multi-band Stacking
This notebook has been heavily optimized to reduce Google Earth Engine API overhead by 700x. Instead of downloading 744 individual hourly files per month, it stacks the entire month into a single request and cleanly unpacks it into a time-dimensioned NetCDF file.


In [1]:
!pip install geemap earthengine-api rioxarray geopandas xarray netCDF4 --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.2 MB/s eta 0:00:00


In [2]:
import ee
import geopandas as gpd
import geemap
from shapely.validation import make_valid
from shapely.ops import transform, unary_union
import os
import rioxarray as rxr
import pandas as pd
import calendar
from datetime import datetime, timedelta

# ==========================================
# 0. PENGATURAN DIREKTORI DINAMIS
# ==========================================
BASE_DIR = os.getcwd()

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    file_geojson = "/kaggle/input/datasets/jerismeteo/projek-downscale/33.05_kecamatan.geojson"
else:
    file_geojson = os.path.join(BASE_DIR, "33.05_kecamatan.geojson")

FOLDER_BASE_OUTPUT = os.path.join(BASE_DIR, "data", "gsmap")

tahun_awal = 2000
tahun_akhir = 2026


In [3]:
import json
from google.oauth2.service_account import Credentials
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    service_account_info = json.loads(user_secrets.get_secret("GEE_KEY"))
    
    SCOPES = ['https://www.googleapis.com/auth/earthengine']
    credentials = Credentials.from_service_account_info(service_account_info, scopes=SCOPES)
    
    ee.Initialize(credentials=credentials, project='staklimjerukagung')
    print("✅ Berhasil Inisialisasi GEE via Service Account (GEE_KEY)")

except Exception as e:
    print(f"⚠️ Gagal Inisialisasi via Secrets, mencoba auth manual: {e}")
    ee.Authenticate()
    ee.Initialize(project='staklimjerukagung')


✅ Berhasil Inisialisasi GEE via Service Account (GEE_KEY)


In [4]:
# ==========================================
# 2. Persiapan Batas Wilayah (GeoJSON Clean)
# ==========================================
if not os.path.exists(file_geojson):
    raise FileNotFoundError(f"File GeoJSON tidak ditemukan di: {file_geojson}")

gdf = gpd.read_file(file_geojson)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

print("Membersihkan geometri GeoJSON yang cacat...")
gdf = gdf[gdf.geometry.notna()].copy()

def _to_2d(geom):
    if geom is None or geom.is_empty: return None
    return transform(lambda x, y, z=None: (x, y), geom)

def _extract_polygonal(geom):
    if geom is None or geom.is_empty: return None
    if geom.geom_type in ("Polygon", "MultiPolygon"): return geom
    if geom.geom_type == "GeometryCollection":
        polys = [g for g in geom.geoms if g.geom_type in ("Polygon", "MultiPolygon")]
        if not polys: return None
        return unary_union(polys)
    return None

def _clean_geom(geom):
    if geom is None or geom.is_empty: return None
    geom = _to_2d(geom)
    geom = make_valid(geom)
    geom = _extract_polygonal(geom)
    if geom is None or geom.is_empty: return None
    geom = geom.buffer(0)
    if geom is None or geom.is_empty: return None
    if not geom.is_valid:
        geom = make_valid(geom)
        geom = _extract_polygonal(geom)
    if geom is None or geom.is_empty or not geom.is_valid: return None
    return geom

gdf["geometry"] = gdf["geometry"].apply(_clean_geom)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
gdf.reset_index(drop=True, inplace=True)

if gdf.empty:
    raise ValueError("Semua geometri tidak valid setelah proses cleaning.")

print("Mengonversi ke Earth Engine...")
geojson_fc = gdf.__geo_interface__
batas_kebumen = ee.FeatureCollection(geojson_fc["features"])


Membersihkan geometri GeoJSON yang cacat...
Mengonversi ke Earth Engine...


In [5]:
# ==========================================
# 3. FUNGSI UNDUH SUPER-CEPAT (MULTI-BAND STACKING)
# ==========================================
def unduh_gsmap_bulanan(tahun, bulan, batas_ee, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    # Tentukan tgl_mulai dan tgl_akhir untuk 1 bulan penuh
    tgl_mulai = f"{tahun}-{bulan:02d}-01T00:00:00"
    if bulan == 12:
        tgl_akhir = f"{tahun+1}-01-01T00:00:00"
    else:
        tgl_akhir = f"{tahun}-{bulan+1:02d}-01T00:00:00"
        
    nc_path = os.path.join(output_dir, f"gsmap_{tahun}_{bulan:02d}.nc")
    tif_path = os.path.join(output_dir, f"gsmap_{tahun}_{bulan:02d}_temp.tif")
    
    if os.path.exists(nc_path):
        print(f"[{tahun}-{bulan:02d}] File sudah ada, dilewati...")
        return
        
    print(f"[{tahun}-{bulan:02d}] Menarik data ImageCollection dari GEE...")
    try:
        # 1. Filter Koleksi GSMaP v8 Operational
        dataset = ee.ImageCollection('JAXA/GPM_L3/GSMaP/v8/operational') \
                    .filter(ee.Filter.date(tgl_mulai, tgl_akhir)) \
                    .select('hourlyPrecipRateGC')
        
        # Hitung jumlah jam (images) yang tersedia
        count = dataset.size().getInfo()
        if count == 0:
            print(f"[{tahun}-{bulan:02d}] ⚠️ Tidak ada data di GEE untuk periode ini.")
            return
            
        # 2. Ekstrak Timestamp Asli Langsung Dari GEE
        # Ini krusial agar dimensi waktu di NetCDF 100% presisi dan berurutan
        timestamps_ms = dataset.aggregate_array('system:time_start').getInfo()
        time_index = pd.to_datetime(timestamps_ms, unit='ms')
        
        # 3. Stack Ratusan Image Menjadi 1 File Multi-band
        stacked_image = dataset.toBands().clip(batas_ee)
        
        print(f"[{tahun}-{bulan:02d}] Mengunduh {count} jam sekaligus ke TIF sementara...")
        geemap.ee_export_image(
            stacked_image,
            filename=tif_path,
            region=batas_ee.geometry(),
            scale=11132,
            file_per_band=False
        )
        
        # 4. Konversi Cerdas ke NetCDF Menggunakan xarray
        print(f"[{tahun}-{bulan:02d}] Konversi TIF Multi-band -> NetCDF Time-Series...")
        with rxr.open_rasterio(tif_path, masked=True) as da:
            # Rasterio memuat dimensi (band, y, x)
            # Kita ubah dimensi 'band' menjadi 'time' dan pasang timestamp dari GEE
            da = da.rename({'band': 'time'})
            
            # Validasi jika jumlah band cocok dengan jumlah timestamp
            if len(da.time) == len(time_index):
                da['time'] = time_index
            else:
                # Fallback aman jika terjadi ketidaksesuaian kecil
                da['time'] = pd.date_range(start=time_index[0], periods=len(da.time), freq='h')
                
            da.name = "precipitation"
            
            # Simpan kompresi NetCDF super efisien
            da.to_netcdf(nc_path)
            
        # Bersihkan file TIF sementara
        os.remove(tif_path)
        print(f"[{tahun}-{bulan:02d}] ✓ Selesai! Tersimpan di {nc_path}\n")
        
    except Exception as e:
        print(f"[{tahun}-{bulan:02d}] ❌ Error: {e}")
        if os.path.exists(tif_path):
            os.remove(tif_path)

def unduh_gsmap_multi_tahun(tahun_awal, tahun_akhir, batas_ee, output_base_dir):
    print(f"\n{'='*60}")
    print(f"UNDUH GSMaP OPTIMASI STACKING: {tahun_awal} - {tahun_akhir}")
    print(f"{'='*60}\n")
    
    for tahun in range(tahun_awal, tahun_akhir + 1):
        folder_tahun = os.path.join(output_base_dir, str(tahun))
        for bulan in range(1, 13):
            unduh_gsmap_bulanan(tahun, bulan, batas_ee, folder_tahun)
            
    print(f"{'='*60}")
    print("SELESAI UNDUH SEMUA TAHUN")
    print(f"{'='*60}")


In [6]:
# ==========================================
# 4. EKSEKUSI FUNGSI UTAMA
# ==========================================
unduh_gsmap_multi_tahun(
    tahun_awal=tahun_awal,
    tahun_akhir=tahun_akhir,
    batas_ee=batas_kebumen,
    output_base_dir=FOLDER_BASE_OUTPUT
)



UNDUH GSMaP OPTIMASI STACKING: 2000 - 2026

[2000-01] Menarik data ImageCollection dari GEE...
[2000-01] Mengunduh 744 jam sekaligus ke TIF sementara...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/gsmap/2000/gsmap_2000_01_temp.tif
[2000-01] Konversi TIF Multi-band -> NetCDF Time-Series...


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[2000-01] ✓ Selesai! Tersimpan di /kaggle/working/data/gsmap/2000/gsmap_2000_01.nc

[2000-02] Menarik data ImageCollection dari GEE...
[2000-02] Mengunduh 696 jam sekaligus ke TIF sementara...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/gsmap/2000/gsmap_2000_02_temp.tif
[2000-02] Konversi TIF Multi-band -> NetCDF Time-Series...


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[2000-02] ✓ Selesai! Tersimpan di /kaggle/working/data/gsmap/2000/gsmap_2000_02.nc

[2000-03] Menarik data ImageCollection dari GEE...
[2000-03] Mengunduh 744 jam sekaligus ke TIF sementara...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/gsmap/2000/gsmap_2000_03_temp.tif
[2000-03] Konversi TIF Multi-band -> NetCDF Time-Series...


...

[2026-05] ✓ Selesai! Tersimpan di /kaggle/working/data/gsmap/2026/gsmap_2026_05.nc

[2026-06] Menarik data ImageCollection dari GEE...
[2026-06] Mengunduh 720 jam sekaligus ke TIF sementara...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/gsmap/2026/gsmap_2026_06_temp.tif
[2026-06] Konversi TIF Multi-band -> NetCDF Time-Series...


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[2026-06] ✓ Selesai! Tersimpan di /kaggle/working/data/gsmap/2026/gsmap_2026_06.nc

[2026-07] Menarik data ImageCollection dari GEE...
[2026-07] Mengunduh 741 jam sekaligus ke TIF sementara...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/gsmap/2026/gsmap_2026_07_temp.tif
[2026-07] Konversi TIF Multi-band -> NetCDF Time-Series...


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[2026-07] ✓ Selesai! Tersimpan di /kaggle/working/data/gsmap/2026/gsmap_2026_07.nc

[2026-08] Menarik data ImageCollection dari GEE...
[2026-08] ⚠️ Tidak ada data di GEE untuk periode ini.
[2026-09] Menarik data ImageCollection dari GEE...
[2026-09] ⚠️ Tidak ada data di GEE untuk periode ini.
[2026-10] Menarik data ImageCollection dari GEE...
[2026-10] ⚠️ Tidak ada data di GEE untuk periode ini.
[2026-11] Menarik data ImageCollection dari GEE...
[2026-11] ⚠️ Tidak ada data di GEE untuk periode ini.
[2026-12] Menarik data ImageCollection dari GEE...
[2026-12] ⚠️ Tidak ada data di GEE untuk periode ini.
SELESAI UNDUH SEMUA TAHUN
